# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive full credit on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**For half credit, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description and feel free to experiment as you see fit.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

In [25]:
import pandas as pd
import altair as alt
import numpy as np

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [26]:
def legislators_df():
    df = pd.read_csv("data/legislators.csv")
    df["state"] = df["jurisdiction"].str.extract(
        "(?:state|district|territory):([a-z]{2})"
    )
    return df

# render dataframe
legislators_df()

,name,given_name,family_name,gender,party,jurisdiction,district,type,state
0,Robert White,Robert C.,White,Male,Independent,ocd-jurisdiction/country:us/district:dc/govern...,At-Large,legislature,dc
1,Charles Allen,Charles,Allen,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 6,legislature,dc
2,Vince Gray,Vincent Condol,Gray,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 7,legislature,dc
3,Phil Mendelson,Philip Heath,Mendelson,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Chairman,legislature,dc
4,Trayon White,Trayon,White,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 8,legislature,dc
...,...,...,...,...,...,...,...,...,...
7430,Sol Higgins,Sol Yamiz,Higgins Cuadrado,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,35,lower,pr
7431,Migdalia González,Migdalia I.,Gonzalez Arroyo,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,4,upper,pr
7432,Keren Riquelme,Keren L.,Riquelme Cabrera,Female,Partido Nuevo Progresista,ocd-jurisdiction/country:us/territory:pr/gover...,At-Large,upper,pr
7433,Jocelyne Rodríguez Negrón,Jocelyne M.,Rodriguez Negron,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,19,lower,pr


### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [27]:
def states_by_gender_initial(df):

    counts = (
        legislators_df()
        .groupby(["state", "gender"])
        .size()
        .reset_index(name="count")
    )
    
    chart = alt.Chart(counts).mark_bar().encode(
        x="state",
        y="sum(count)",
        color="gender"
    )
    return chart

# render chart
states_by_gender_initial(legislators_df())

alt.Chart(...)

### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [28]:
def states_by_gender_improved(df):

    counts = (
        legislators_df()
        .groupby(["state", "gender"])
        .size()
        .reset_index(name="count")
    )
    
    chart = alt.Chart(counts).mark_bar().encode(
        x='state',
        y=alt.Y(
            "sum(count):Q",
            stack="normalize",
            axis=alt.Axis(format="%", title="Share of legislators")
        ),
        color=alt.Color(
            "gender:N",
            scale=alt.Scale(
                domain=["Male", "Other", "Female"],
                range=["#a4a3aa", "#353148", "#D6A35D"]
            )
        )
    )

    threshold_line = (
        alt.Chart()
        .mark_rule(color="black")
        .encode(y=alt.datum(0.5))
    )

    return chart + threshold_line

# render chart
states_by_gender_improved(legislators_df())

alt.LayerChart(...)

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [29]:
def party_control_raw(df):

    counts = (
        legislators_df()
        .groupby(["state", "party"])
        .size()
        .reset_index(name="count")
    )
    
    chart = alt.Chart(counts).mark_bar().encode(
        x='state',
        y=alt.Y(
            "sum(count):Q",
            stack="normalize",
            axis=alt.Axis(format="%", title="Share of legislators by party")
        ),
        color="party"
    )

    threshold_line = (
        alt.Chart()
        .mark_rule(color="black")
        .encode(y=alt.datum(0.5))
    )

    return chart + threshold_line

# render chart
party_control_raw(legislators_df())

alt.LayerChart(...)

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [30]:
def clean_party_df():
    # start with the DataFrame from part 1 & return transformed copy
    df = legislators_df()

    df["party_code"] = np.select(
        [
            df["party"].str.contains("Democratic", na=False),
            df["party"].str.contains("Republican", na=False)
        ],
        ["D", "R"],
        default="O"
    )

    df = df[~df["state"].isin(["dc", "ne", "pr"])]

    return df

# render dataframe
clean_party_df()

,name,given_name,family_name,gender,party,jurisdiction,district,type,state,party_code
13,Kevin McCabe,Kevin J.,McCabe,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,30,lower,ak,R
14,C.J. McCormick,Conrad J.,McCormick,Male,Democratic,ocd-jurisdiction/country:us/state:ak/government,38,lower,ak,D
15,Bill Wielechowski,Bill P.,Wielechowski,Male,Democratic,ocd-jurisdiction/country:us/state:ak/government,K,upper,ak,D
16,Justin Ruffridge,Justin,Ruffridge,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,7,lower,ak,R
17,Will Stapp,Will,Stapp,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,32,lower,ak,R
...,...,...,...,...,...,...,...,...,...,...
7353,Karlee Provenza,Karlee R.,Provenza,Female,Democratic,ocd-jurisdiction/country:us/state:wy/government,45,lower,wy,D
7354,Affie Ellis,Affie Burnside,Ellis,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,8,upper,wy,R
7355,Abby Angelos,Abby,Angelos,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,3,lower,wy,R
7356,Pepper Ottman,Pepper L.,Ottman,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,34,lower,wy,R


### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [31]:
def party_control_by_chamber(df):

    df = df[df["type"].isin(["upper", "lower"])]

    df = (
        df.groupby(["state", "type", "party_code"])
        .size()
        .reset_index(name="count")
    )

    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X(
            "sum(count):Q",
            stack="normalize",
            axis=alt.Axis(format="%", title="Share of Legislators")
        ),
        y="state:N",
        color="party_code:N"
    )

    threshold_line = (
        alt.Chart()
        .mark_rule(color="black")
        .encode(x=alt.datum(0.5))
    )

    combined = alt.layer(chart, threshold_line, data=df)

    return combined.facet(
        column=alt.Column("type:N", sort=["lower", "upper"])
    )

party_control_by_chamber(clean_party_df())

alt.FacetChart(...)

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [32]:
def population_combined_df():

    df_pop = pd.read_csv("data/populations.csv")
    df = legislators_df()

    counts = (
        df[df["type"].isin(["upper", "lower"])]
        .groupby(["state", "type"])
        .size()
        .unstack(fill_value=0)
    )

    df_combined = df_pop.merge(
        counts,
        on="state",
        how="left"
    )

    df_combined = df_combined.dropna(subset=["lower", "upper"]).reset_index()

    return df_combined

population_combined_df()

,index,state,pop_2020,lower,upper
0,0,al,5024294,104.0,35.0
1,1,ak,733374,40.0,20.0
2,2,az,7157902,59.0,30.0
3,3,ar,3011490,98.0,35.0
4,4,ca,39538212,79.0,40.0
5,5,co,5773707,65.0,35.0
6,6,ct,3605912,151.0,36.0
7,7,de,989946,41.0,21.0
8,9,fl,21538216,119.0,40.0
9,10,ga,10713771,180.0,56.0


### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [33]:
def scatter_pop_size():

    df = population_combined_df()
        
    upper_chart = alt.Chart(df).mark_point(
        shape="triangle-up",
        color="purple",
        size=60
    ).encode(
        x=alt.X("pop_2020:Q", title="Population (2020)"),
        y=alt.Y("upper:Q", title="Number of Legislative Seats"),
        tooltip=["state:N", "pop_2020:Q", "upper:Q"]
    )

    lower_chart = alt.Chart(df).mark_point(
        shape="triangle-down",
        color="orange",
        size=60
    ).encode(
        x=alt.X("pop_2020:Q", title="Population (2020)"),
        y=alt.Y("lower:Q", title="Number of Legislative Seats"),
        tooltip=["state:N", "pop_2020:Q", "lower:Q"]
    )
    return (
        upper_chart + lower_chart
    ).properties(
            width=600,
            height=400
    ).interactive()

scatter_pop_size()

alt.LayerChart(...)

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [34]:
def scatter_pop_size_regression():

    df = population_combined_df()
        
    upper_chart = alt.Chart(df).mark_point(
        shape="triangle-up",
        color="purple",
        size=60
    ).encode(
        x=alt.X("pop_2020:Q", title="Population (2020)"),
        y=alt.Y("upper:Q", title="Number of Legislative Seats"),
        tooltip=["state:N", "pop_2020:Q", "upper:Q"]
    )

    lower_chart = alt.Chart(df).mark_point(
        shape="triangle-down",
        color="orange",
        size=60
    ).encode(
        x=alt.X("pop_2020:Q", title="Population (2020)"),
        y=alt.Y("lower:Q", title="Number of Legislative Seats"),
        tooltip=["state:N", "pop_2020:Q", "upper:Q"]
    )

    upper_regression = alt.Chart(df).transform_regression(
        "pop_2020", "upper"
    ).mark_line(
        color="purple",
        size=1
    ).encode(
        x="pop_2020:Q",
        y="upper:Q"
    )

    lower_regression = alt.Chart(df).transform_regression(
        "pop_2020", "lower"
    ).mark_line(
        color="orange",
        size=1
    ).encode(
        x="pop_2020:Q",
        y="lower:Q"
    )

    return (
        upper_chart
        + lower_chart
        + upper_regression
        + lower_regression
    ).properties(
        width=600,
        height=400
    ).interactive()

scatter_pop_size_regression()

alt.LayerChart(...)

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [35]:
def actions_df():

    df_actions = pd.read_csv("data/actions_il-in-mi-wi_2021-2024.csv")
    df_actions["date"] = df_actions["date"].str.extract(r'(\d{4}-\d{2}-\d{2})')
    df_actions["date"] = pd.to_datetime(df_actions["date"])

    return df_actions

actions_df()

,description,date,classification,state,session
0,"Filed with the Clerk by Rep. Marcus C. Evans, Jr.",2021-02-10,['filing'],IL,102nd
1,Filed with Secretary,2021-08-26,['filing'],IL,102nd
2,Filed with Secretary by Sen. Jacqueline Y. Col...,2021-02-25,['filing'],IL,102nd
3,Filed with Secretary,2022-01-19,['filing'],IL,102nd
4,Filed with Secretary,2022-11-29,['filing'],IL,102nd
...,...,...,...,...,...
456400,Partial veto failed to pass notwithstanding th...,2022-05-17,[],WI,2021
456401,Fiscal estimate received,2021-02-08,['receipt'],WI,2021
456402,Fiscal estimate received,2021-02-10,['receipt'],WI,2021
456403,Placed on calendar 5-17-2022 pursuant to Joint...,2022-05-17,[],WI,2021


### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [36]:
def actions_heatmap_scaled(df):

    df = df.copy()

    # Create a column identifying each week
    df["week"] = df["date"].dt.to_period("W").dt.start_time

    # Count actions for each state and week
    counts = (
        df.groupby(["state", "week"])
        .size()
        .reset_index(name="count")
    )

    counts["week_end"] = counts["week"] + pd.Timedelta(days=7)

    chart = alt.Chart(counts).mark_rect().encode(
        x=alt.X(
            "week:T",
            title="date (year-week)",
            axis=alt.Axis(
                format="%Y W%U",
                tickCount=15,
                labelAngle=0,
            )
        ),
        x2="week_end:T",
        y=alt.Y("state:N", title="State"),
        color=alt.Color("count:Q", title="Number of Actions")
    ).properties(
        width=1200,
        height=200
    )

    return chart

actions_heatmap_scaled(actions_df())

alt.Chart(...)

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [37]:
# MODIFY THIS LINE (exclude IL)
actions_heatmap_scaled(actions_df()[actions_df()["state"] != "IL"])

alt.Chart(...)

In [38]:
# MODIFY THIS LINE (only IL)
actions_heatmap_scaled(actions_df()[actions_df()["state"] == "IL"])

alt.Chart(...)

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [39]:
def actions_cumulative(df):

    df = df.copy()

    df["day"] = df["date"]

    counts = (
        df.groupby(["state", "day"])
        .size()
        .reset_index(name="count")
    )
    tick_dates = [
        "2021-01-04",
        "2021-06-22",
        "2021-12-21",
        "2022-06-16",
        "2022-12-31",
        "2023-06-22",
        "2023-12-21",
        "2024-06-17"
    ]

    chart = (
        alt.Chart(counts)
        .transform_window(
            cumulative_actions="sum(count)",
            sort=[alt.SortField("day")],
            groupby=['state']
        )
        .mark_line()
        .encode(
            x=alt.X(
                "day:T",
                title="Date (year-month-day)",
                scale=alt.Scale(domain=["2021-01-04", "2024-09-25"]),
                axis=alt.Axis(
                    values=tick_dates,
                    format="%Y-%m-%d",
                )
            ),

            y=alt.Y("cumulative_actions:Q", title="cumulative actions"),
            color=alt.Color("state:N", title="State")
        )
        .properties(
            width=720,
            height=450
        )
    )
    return chart

actions_cumulative(actions_df())

alt.Chart(...)

## Part 5. Replicating a Chart

Find a chart (perhaps the one you're using for your first Analysis post!) and do your best to replicate it with Altair. Since you likely won't have access to the raw data-- start by mocking up a dataset that is similar enough to what your chart uses, don't worry about exact values, just that the correct attributes are present with some variation.

Your replicated chart should use similar colors, labels, etc. to the one that you chose to reproduce. Again, we aren't looking for a pixel-perfect replica, but something that pushes you to explore parts of Altair's documentation/features that you didn't use above.


In [40]:
source = pd.read_csv("data/laborshare.csv")

countries = ["Canada", "Germany", "France", "US",
             "UK", "Australia", "Japan", "Italy"]

colors = ["#527C9C", "#888888", "#B00048", "#203746",
          "#12607B", "#713A92", "#F29A22", "#38A56B"]

color = alt.Color(
    "country:N",
    scale=alt.Scale(domain=countries, range=colors),
    legend=None
)

line = alt.Chart(source).mark_line().encode(
    x=alt.X(
        "year:Q",
        scale=alt.Scale(domain=[1970, 2019]),
        axis=alt.Axis(format="d")
    ),
    y=alt.Y(
        "labor_share:Q",
        scale=alt.Scale(domain=[45, 80]),
        axis=alt.Axis(values=[45, 50, 55, 60, 65, 70, 75, 80])
    ),
    color=color,
    size=alt.condition("datum.country == 'US'", alt.value(4), alt.value(2))
)

last = source.sort_values("year").groupby("country").tail(1)

labels = alt.Chart(last[last["country"] != "US"]).mark_text(
    align="left",
    dx=5,
    fontSize=12
).encode(
    x="year:Q",
    y="labor_share:Q",
    text="country:N",
    color=color
)

us_label = alt.Chart(last[last["country"] == "US"]).mark_text(
    align="left",
    dx=5,
    fontSize=12,
    fontWeight="bold"
).encode(
    x="year:Q",
    y="labor_share:Q",
    text="country:N",
    color=color
)

chart = (line + labels + us_label).properties(
    width=720,
    height=480,
    title="Labor compensation as percent of GDP, 1970–2019"
)

chart

alt.LayerChart(...)